In [ ]:
import ee
import os
import pandas as pd
import matplotlib.pyplot as plt
import hydropandas as hpd
from python_projects.beroepsproduct.tsconfig import START_DATE, END_DATE, extent_to_rd, extent_to_ee, NDVI_START_DATE, NDVI_END_DATE
import numpy as np
import folium
import matplotlib.patches as mpatches

hpd.util.get_color_logger("INFO")
# pd.set_option("display.max_rows", None)
# pd.set_option("display.max_columns", None)
# pd.set_option("display.width", None)

In [ ]:
organisation = "rotterdam"

auth = ("__key__", "D5aclEis.RBUeIvKA6jrLVUzNpPATjvGyLXsLAx7P")

In [ ]:
my_extent = extent_to_rd("gw") 
oc = hpd.read_lizard(
    extent=my_extent,
    which_timeseries=["hand", "diver", "diver_validated"],
    datafilters=None,
    combine_method="merge",
    organisation=organisation,
    auth=auth,
)

In [ ]:
m = oc.plots.interactive_map(
    color="red",
    zoom_start=20,
    tiles="Esri.WorldImagery",
    popup_width=350
)

lon_min, lat_min, lon_max, lat_max = extent_to_ee("ndvi")

folium.Rectangle(
    bounds=[(lat_min, lon_min), (lat_max, lon_max)],  # folium gebruikt (lat, lon)
    fill=False,
    weight=3
).add_to(m)

m

In [ ]:
oc

In [ ]:
gw = oc.obs["GMW000000046935001"]
print(gw)

In [ ]:
ts = gw["value"].copy()
ts.index = pd.to_datetime(ts.index)
ts = ts.dropna()

print(ts.index.min(), ts.index.max())
print(ts.shape)

In [ ]:
ts_year = ts.loc[START_DATE:END_DATE]

ts_daily_raw = ts_year.resample("D").mean()

ts_daily = ts_daily_raw.interpolate(method="time")

is_interpolated = ts_daily_raw.isna()

In [ ]:
percentile = 0.30

# === FIXED THRESHOLD ===
threshold_fixed = ts_daily.quantile(percentile)

# === VARIABLE THRESHOLD ===
monthly_threshold = (
    ts_daily
    .groupby(ts_daily.index.month)
    .quantile(percentile)
    .reindex(range(1, 13))          # zorg dat alle maanden bestaan
)

n_months_with_data = monthly_threshold.notna().sum()

if n_months_with_data < 6:
    print(
        f"Not enough data for variable threshold ({n_months_with_data} months). "
        "Variable threshold will not be used."
    )
    threshold_variable_smooth = None
else:
    monthly_threshold = monthly_threshold.interpolate(method="linear")

    threshold_variable = ts_daily.index.to_series().map(
        lambda d: monthly_threshold.loc[d.month]
    )

    threshold_variable_smooth = (
        threshold_variable
        .rolling(window=20, min_periods=1)
        .mean()
    )

# === KIES welke threshold wordt gebruikt voor DROOGTE ===
threshold = threshold_variable_smooth if threshold_variable_smooth is not None else threshold_fixed


In [ ]:
is_drought_fixed = ts_daily <= threshold_fixed

if threshold_variable_smooth is not None:
    is_drought_variable = ts_daily <= threshold_variable_smooth
else:
    is_drought_variable = None

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(
    ts_daily.index,
    ts_daily,
    color="steelblue",
    linewidth=1.5,
    label="daily groundwater level",
)

ax.axhline(
    threshold_fixed,
    color="orange",
    linewidth=2,
    linestyle="-",
    label="fixed threshold (30th percentile)",
)

ax.fill_between(
    ts_daily.index,
    ts_daily,
    threshold_fixed,
    where=is_drought_fixed,
    color="red",
    alpha=0.5,
    label="drought event (fixed)",
    interpolate=True,
)

ax.set_ylabel("Groundwater level (m NAP)")
ax.set_xlabel("Date")
ax.set_title(f"Groundwater drought – FIXED threshold\n{gw.name}")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
if threshold_variable_smooth is not None:
    fig, ax = plt.subplots(figsize=(10, 4))

    ax.plot(
        ts_daily.index,
        ts_daily,
        color="steelblue",
        linewidth=1.5,
        label="daily groundwater level",
    )

    ax.plot(
        threshold_variable_smooth.index,
        threshold_variable_smooth.values,
        color="orange",
        linewidth=2,
        label="variable threshold (monthly + smoothed)",
    )

    ax.fill_between(
        ts_daily.index,
        ts_daily,
        threshold_variable_smooth,
        where=is_drought_variable,
        color="red",
        alpha=0.5,
        label="drought event (variable)",
        interpolate=True,
    )

    ax.set_ylabel("Groundwater level (m NAP)")
    ax.set_xlabel("Date")
    ax.set_title(f"Groundwater drought – VARIABLE threshold\n{gw.name}")
    ax.legend()

    plt.tight_layout()
    plt.show()
else:
    print("Variable threshold plot skipped: insufficient data.")


In [ ]:
# =========================
# SAMENVATTING – FIXED THRESHOLD
# =========================
duration_fixed = is_drought_fixed.sum()
max_deficit_fixed = (threshold_fixed - ts_daily[is_drought_fixed]).max()

n_interpolated = is_interpolated.sum()

print("=== FIXED THRESHOLD (30th percentile) ===")
print(
    f"Drought duration (2024–2025): {duration_fixed} days. "
    f"{n_interpolated} days were interpolated."
)
print(f"Maximum deficit (2024–2025): {max_deficit_fixed:.2f} m\n")


# =========================
# SAMENVATTING – VARIABLE THRESHOLD
# =========================
if is_drought_variable is not None:
    duration_variable = is_drought_variable.sum()
    max_deficit_variable = (
        threshold_variable_smooth - ts_daily[is_drought_variable]
    ).max()

    print("=== VARIABLE THRESHOLD (monthly 30th percentile + smoothing) ===")
    print(
        f"Drought duration (2024–2025): {duration_variable} days. "
        f"{n_interpolated} days were interpolated."
    )
    print(f"Maximum deficit (2024–2025): {max_deficit_variable:.2f} m\n")
else:
    print("=== VARIABLE THRESHOLD ===")
    print("Not enough data to compute variable threshold.\n")


# =========================
# DAGELIJKS OVERZICHT (WAARDES + HERKOMST)
# =========================
origin_per_day = (
    gw[["value", "origin"]]
    .dropna()
    .assign(date=lambda df: df.index.normalize())
    .groupby("date")["origin"]
    .agg(lambda x: ",".join(sorted(set(x))))
)

summary = pd.DataFrame({
    "value": ts_daily,
    "interpolated": is_interpolated,
})

summary["origin"] = origin_per_day
summary.loc[summary["interpolated"], "origin"] = "interpolated"

# Toon de volledige tabel (alle rijen en kolommen)
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 2000):
    print(summary.to_string())



In [ ]:
# =========================
# COMPACTE & SNELLE EE INIT
# =========================

project = "afstuderen-481613"

try:
    ee.Initialize(project=project)
    print("Earth Engine initialized.")
except Exception:
    ee.Authenticate()
    ee.Initialize(project=project)
    print("Earth Engine authenticated and initialized.")

roi = ee.Geometry.Rectangle(extent_to_ee("ndvi"))

In [ ]:
def mask_s2_clouds(img):
    scl = img.select("SCL")
    mask = (
        scl.neq(3)   # cloud shadow
        .And(scl.neq(7))
        .And(scl.neq(8))
        .And(scl.neq(9))   # clouds
        .And(scl.neq(10))
        .And(scl.neq(11))  # snow
    )
    return img.updateMask(mask)


def add_ndvi_s2(img):
    return img.addBands(
        img.normalizedDifference(["B8", "B4"]).rename("NDVI")
    )


def reduce_ndvi(img):
    stats = img.select("NDVI").reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.count(), None, True),
        geometry=roi,
        scale=10,
        bestEffort=True,
        maxPixels=1e13,
    )

    return ee.Feature(None, {
        "date": ee.Date(img.get("system:time_start")).format("YYYY-MM-dd"),
        "ndvi": stats.get("NDVI_mean"),
        "n_pixels": stats.get("NDVI_count"),
    })


col = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(roi)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
    .map(mask_s2_clouds)
    .map(add_ndvi_s2)
)

fc = col.map(reduce_ndvi)
features = fc.getInfo()["features"]

rows = []
for f in features:
    p = f["properties"]
    if p["ndvi"] is not None and p["n_pixels"] is not None:
        rows.append(p)

df_s2 = pd.DataFrame(rows)
df_s2["date"] = pd.to_datetime(df_s2["date"])
df_s2 = df_s2.sort_values("date").reset_index(drop=True)

In [ ]:
roi_area = roi.area().getInfo()
expected_pixels = roi_area / (10 * 10)
min_required = expected_pixels * 0.70

df_s2 = df_s2[df_s2["n_pixels"] >= min_required].reset_index(drop=True)

print("Remaining NDVI scenes:", len(df_s2))

In [ ]:
# Use NDVI-specific start and end dates (all caps)
NDVI_START_DATE = pd.to_datetime(NDVI_START_DATE)
NDVI_END_DATE = pd.to_datetime(NDVI_END_DATE)
ndvi_ts = df_s2.set_index("date")["ndvi"]
ndvi_daily_raw = ndvi_ts.loc[NDVI_START_DATE:NDVI_END_DATE].resample("D").mean()
ndvi_daily = ndvi_daily_raw.interpolate("time")


In [ ]:
# Calculate NDVI threshold for NDVI-specific period (all caps)
ndvi_threshold_fixed = ndvi_daily.loc[NDVI_START_DATE:NDVI_END_DATE].quantile(0.30)
is_ndvi_stress = ndvi_daily <= ndvi_threshold_fixed


In [ ]:
# Plot NDVI daily values with the 30th percentile threshold (calculated for period between NDVI_START_DATE and NDVI_END_DATE)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ndvi_daily.index, ndvi_daily, label='NDVI', color='green')
ax.axhline(ndvi_threshold_fixed, color='orange', linestyle='--', linewidth=2, label='30th percentile (NDVI period)')
ax.set_xlabel('Date')
ax.set_ylabel('NDVI')
ax.set_title('NDVI Daily Values with 30th Percentile Threshold')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# =========================
# TWO TIMELINES: FIXED GW vs VARIABLE GW (both with fixed NDVI)
# =========================
# Fixed NDVI
# Build a common daily index that covers the available NDVI and groundwater ranges
# This makes the timeline code robust to different extents and date ranges.
indices = [i for i in [ts_daily.index, ndvi_daily.index] if len(i) > 0]
# Force plot to start at START_DATE and end at END_DATE
plot_start = pd.to_datetime(START_DATE)
plot_end = pd.to_datetime(END_DATE)
if len(indices) == 0:
    common_index = pd.DatetimeIndex([])
else:
    start = min(idx.min() for idx in indices)
    end = max(idx.max() for idx in indices)
    try:
        start_limit = plot_start
        end_limit = plot_end
        start = max(pd.to_datetime(start), start_limit)
        end = min(pd.to_datetime(end), end_limit)
    except Exception:
        pass
    # Always start at START_DATE and end at END_DATE
    common_index = pd.date_range(start=plot_start.normalize(), end=plot_end.normalize(), freq="D")
# Fixed NDVI
ndvi_fixed = is_ndvi_stress.reindex(common_index).fillna(False)
# Fixed groundwater
gw_fixed = is_drought_fixed.reindex(common_index).fillna(False)
# Variable groundwater
ts_daily_common = ts_daily.reindex(common_index)
monthly_threshold_gw = (
    ts_daily_common
    .groupby(ts_daily_common.index.month)
    .quantile(percentile)
    .reindex(range(1, 13))
    .interpolate(method="linear")
)
threshold_variable_gw = ts_daily_common.index.to_series().map(
    lambda d: monthly_threshold_gw.loc[d.month]
)
threshold_variable_gw_smooth = (
    threshold_variable_gw
    .rolling(window=20, min_periods=1)
    .mean()
)
gw_variable = (ts_daily_common <= threshold_variable_gw_smooth).fillna(False)
# --- Combinaties voor Fixed GW + Fixed NDVI ---
both_fixed = ndvi_fixed & gw_fixed
ndvi_only_fixed = ndvi_fixed & ~gw_fixed
gw_only_fixed = gw_fixed & ~ndvi_fixed
neither_fixed = ~(ndvi_fixed | gw_fixed)
# --- Combinaties voor Variable GW + Fixed NDVI ---
both_var = ndvi_fixed & gw_variable
ndvi_only_var = ndvi_fixed & ~gw_variable
gw_only_var = gw_variable & ~ndvi_fixed
neither_var = ~(ndvi_fixed | gw_variable)
# --- No data mask ---
no_data_mask = ts_daily_common.isna() | ndvi_fixed.isna()
# Functie om segmenten te bouwen
def build_segments(both, gw_only, ndvi_only, no_data):
    timeline = pd.Series("neither", index=common_index)
    timeline.loc[both] = "both"
    timeline.loc[gw_only] = "gw_only"
    timeline.loc[ndvi_only] = "ndvi_only"
    timeline.loc[no_data] = "no_data"
    blocks = timeline.ne(timeline.shift()).cumsum()
    return timeline.groupby(blocks).agg(
        start=lambda x: x.index[0],
        end=lambda x: x.index[-1],
        category="first"
    )
colors = {
    "both": "red",
    "gw_only": "blue",
    "ndvi_only": "green",
    "neither": "lightgrey",
    "no_data": "black",
}
segments_fixed = build_segments(both_fixed, gw_only_fixed, ndvi_only_fixed, no_data_mask)
segments_var = build_segments(both_var, gw_only_var, ndvi_only_var, no_data_mask)
fig, ax = plt.subplots(figsize=(14, 3))
# Y-posities en bar-height
y_fixed = 0.6
y_var = 0.2
bar_height = 0.18
# Plot Fixed GW + Fixed NDVI
for _, row in segments_fixed.iterrows():
    bar_end = min(row["end"], plot_end.normalize())
    bar_length = (bar_end - row["start"]).days + 1
    ax.barh(
        y_fixed,
        bar_length,
        left=row["start"],
        color=colors[row["category"]],
        height=bar_height
    )
# Plot Variable GW + Fixed NDVI
for _, row in segments_var.iterrows():
    bar_end = min(row["end"], plot_end.normalize())
    bar_length = (bar_end - row["start"]).days + 1
    ax.barh(
        y_var,
        bar_length,
        left=row["start"],
        color=colors[row["category"]],
        height=bar_height
    )
ax.set_yticks([y_fixed, y_var])
ax.set_yticklabels([
    "NDVI Fixed + GW Fixed",
    "NDVI Fixed + GW Variable"
])
ax.set_ylim(0, 0.85)
ax.set_xlim(plot_start.normalize(), plot_end.normalize())
ax.set_xlabel("Date")
ax.set_title("Timeline of NDVI Stress and Groundwater Drought")
# Legenda rechts buiten de plot
# Ensure mpatches is available in this execution context
if 'mpatches' not in globals():
    import matplotlib.patches as mpatches
legend = [
    mpatches.Patch(color="red", label="Both (NDVI + GW)"),
    mpatches.Patch(color="blue", label="GW drought only"),
    mpatches.Patch(color="green", label="NDVI stress only"),
    mpatches.Patch(color="lightgrey", label="Neither"),
    mpatches.Patch(color="black", label="No data"),
    ]
ax.legend(
    handles=legend,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False
    )
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# SEPARATE TIMELINES: FIXED GW + NDVI and VARIABLE GW + NDVI
# =========================
# Build a common daily index that covers the available NDVI and groundwater ranges
indices = [i for i in [ts_daily.index, ndvi_daily.index] if len(i) > 0]
plot_start = pd.to_datetime(START_DATE)
plot_end = pd.to_datetime(END_DATE)
if len(indices) == 0:
    common_index = pd.DatetimeIndex([])
else:
    common_index = pd.date_range(start=plot_start.normalize(), end=plot_end.normalize(), freq="D")
# Fixed NDVI
ndvi_fixed = is_ndvi_stress.reindex(common_index).fillna(False)
# Fixed groundwater
gw_fixed = is_drought_fixed.reindex(common_index).fillna(False)
# Variable groundwater
ts_daily_common = ts_daily.reindex(common_index)
monthly_threshold_gw = (
    ts_daily_common
    .groupby(ts_daily_common.index.month)
    .quantile(percentile)
    .reindex(range(1, 13))
    .interpolate(method="linear")
)
threshold_variable_gw = ts_daily_common.index.to_series().map(
    lambda d: monthly_threshold_gw.loc[d.month]
)
threshold_variable_gw_smooth = (
    threshold_variable_gw
    .rolling(window=20, min_periods=1)
    .mean()
)
gw_variable = (ts_daily_common <= threshold_variable_gw_smooth).fillna(False)
# --- Combinaties voor Fixed GW + Fixed NDVI ---
both_fixed = ndvi_fixed & gw_fixed
ndvi_only_fixed = ndvi_fixed & ~gw_fixed
gw_only_fixed = gw_fixed & ~ndvi_fixed
neither_fixed = ~(ndvi_fixed | gw_fixed)
# --- Combinaties voor Variable GW + Fixed NDVI ---
both_var = ndvi_fixed & gw_variable
ndvi_only_var = ndvi_fixed & ~gw_variable
gw_only_var = gw_variable & ~ndvi_fixed
neither_var = ~(ndvi_fixed | gw_variable)
# --- No data mask ---
no_data_mask = ts_daily_common.isna() | ndvi_fixed.isna()
# Functie om segmenten te bouwen
def build_segments(both, gw_only, ndvi_only, no_data):
    timeline = pd.Series("neither", index=common_index)
    timeline.loc[both] = "both"
    timeline.loc[gw_only] = "gw_only"
    timeline.loc[ndvi_only] = "ndvi_only"
    timeline.loc[no_data] = "no_data"
    blocks = timeline.ne(timeline.shift()).cumsum()
    return timeline.groupby(blocks).agg(
        start=lambda x: x.index[0],
        end=lambda x: x.index[-1],
        category="first"
    )
colors = {
    "both": "red",
    "gw_only": "blue",
    "ndvi_only": "green",
    "neither": "lightgrey",
    "no_data": "black",
}
# --- Plot Fixed GW + Fixed NDVI ---
segments_fixed = build_segments(both_fixed, gw_only_fixed, ndvi_only_fixed, no_data_mask)
fig_fixed, ax_fixed = plt.subplots(figsize=(14, 2.5))
y_fixed = 0.5
bar_height = 0.22
for _, row in segments_fixed.iterrows():
    bar_end = min(row["end"], plot_end.normalize())
    bar_length = (bar_end - row["start"]).days + 1
    ax_fixed.barh(
        y_fixed,
        bar_length,
        left=row["start"],
        color=colors[row["category"]],
        height=bar_height
    )
ax_fixed.set_yticks([y_fixed])
ax_fixed.set_yticklabels(["NDVI Fixed + GW Fixed"])
ax_fixed.set_ylim(0, 1)
ax_fixed.set_xlim(plot_start.normalize(), plot_end.normalize())
ax_fixed.set_xlabel("Date")
ax_fixed.set_title("Timeline of NDVI Stress and Groundwater Drought (Fixed GW)")
if 'mpatches' not in globals():
    import matplotlib.patches as mpatches
legend_fixed = [
    mpatches.Patch(color="red", label="Both (NDVI + GW)"),
    mpatches.Patch(color="blue", label="GW drought only"),
    mpatches.Patch(color="green", label="NDVI stress only"),
    mpatches.Patch(color="lightgrey", label="Neither"),
    mpatches.Patch(color="black", label="No data"),
    ]
ax_fixed.legend(
    handles=legend_fixed,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False
    )
plt.tight_layout()
plt.show()
# --- Plot Variable GW + Fixed NDVI ---
segments_var = build_segments(both_var, gw_only_var, ndvi_only_var, no_data_mask)
fig_var, ax_var = plt.subplots(figsize=(14, 2.5))
y_var = 0.5
for _, row in segments_var.iterrows():
    bar_end = min(row["end"], plot_end.normalize())
    bar_length = (bar_end - row["start"]).days + 1
    ax_var.barh(
        y_var,
        bar_length,
        left=row["start"],
        color=colors[row["category"]],
        height=bar_height
    )
ax_var.set_yticks([y_var])
ax_var.set_yticklabels(["NDVI Fixed + GW Variable"])
ax_var.set_ylim(0, 1)
ax_var.set_xlim(plot_start.normalize(), plot_end.normalize())
ax_var.set_xlabel("Date")
ax_var.set_title("Timeline of NDVI Stress and Groundwater Drought (Variable GW)")
legend_var = [
    mpatches.Patch(color="red", label="Both (NDVI + GW)"),
    mpatches.Patch(color="blue", label="GW drought only"),
    mpatches.Patch(color="green", label="NDVI stress only"),
    mpatches.Patch(color="lightgrey", label="Neither"),
    mpatches.Patch(color="black", label="No data"),
    ]
ax_var.legend(
    handles=legend_var,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False
    )
plt.tight_layout()
plt.show()